# Классификация тарифов: Smart vs Ultra

Строим модель классификации для предсказания тарифа (`is_ultra`) и добиваемся `accuracy >= 0.75`.
Используем ансамблевые методы и подбираем гиперпараметры для `RandomForestClassifier`.

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, ExtraTreesClassifier, VotingClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split, GridSearchCV

In [ ]:
RANDOM_STATE = 12345
DATA_PATH = 'users_behavior.csv'
TARGET = 'is_ultra'

data = pd.read_csv(DATA_PATH)
X = data.drop(columns=[TARGET])
y = data[TARGET]

# 60% train, 20% valid, 20% test
X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_full, y_train_full, test_size=0.25, random_state=RANDOM_STATE, stratify=y_train_full
)

print(f'Train size: {len(X_train)}')
print(f'Valid size: {len(X_valid)}')
print(f'Test size: {len(X_test)}')

In [ ]:
# Подбор гиперпараметров для RandomForest
rf = RandomForestClassifier(random_state=RANDOM_STATE)
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 8, 12, 16],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

rf_search = GridSearchCV(
    rf,
    param_grid=param_grid,
    scoring='accuracy',
    cv=5,
    n_jobs=-1
)
rf_search.fit(X_train, y_train)
best_rf = rf_search.best_estimator_

print('Best RF params:', rf_search.best_params_)
print('Best CV score:', round(rf_search.best_score_, 4))

In [ ]:
# Обучение ансамблей и сравнение на validation
gb = GradientBoostingClassifier(random_state=RANDOM_STATE)
et = ExtraTreesClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1)

models = {
    'RandomForest (tuned)': best_rf,
    'GradientBoosting': gb,
    'ExtraTrees': et
}

valid_scores = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    pred_valid = model.predict(X_valid)
    score = accuracy_score(y_valid, pred_valid)
    valid_scores[name] = score
    print(f'{name}: validation accuracy = {score:.4f}')

voting = VotingClassifier(
    estimators=[('rf', best_rf), ('gb', gb), ('et', et)],
    voting='hard'
)
voting.fit(X_train, y_train)
voting_valid = accuracy_score(y_valid, voting.predict(X_valid))
valid_scores['VotingClassifier'] = voting_valid
print(f'VotingClassifier: validation accuracy = {voting_valid:.4f}')

best_name = max(valid_scores, key=valid_scores.get)
print('Best model on validation:', best_name)

In [ ]:
# Финальная проверка на тесте
final_model = voting if best_name == 'VotingClassifier' else models[best_name]
final_model.fit(X_train_full, y_train_full)

test_pred = final_model.predict(X_test)
test_accuracy = accuracy_score(y_test, test_pred)

print(f'Final test accuracy ({best_name}) = {test_accuracy:.4f}')
print('Target reached:', test_accuracy >= 0.75)

Если `Target reached: True`, требование задачи выполнено.
При необходимости можно расширить `param_grid` или добавить `RandomizedSearchCV`.